# Mashi TTS — Fine-tune XTTS v2
Run on Google Colab with GPU. Change TRACK to switch between 'manual' and 'auto'.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

DRIVE_ROOT = '/content/drive/MyDrive/mashi-tts-bootstrap'
TRACK      = 'manual'   # change to 'auto' for Track A
SEG_DIR    = f'{DRIVE_ROOT}/data/{TRACK}_segments'
MODEL_DIR  = f'{DRIVE_ROOT}/models/checkpoints/{TRACK}'

In [ ]:
!pip install -q TTS==0.22.0

In [ ]:
# Fine-tune XTTS v2 on the Mashi clips
from TTS.demos.xtts_ft_demo.xtts_demo import train_gpt

train_gpt(
    language='sw',                          # Swahili — closest Bantu in XTTS
    num_epochs=10,
    batch_size=4,
    grad_acumm=2,
    train_csv=f'{SEG_DIR}/metadata.csv',
    eval_csv=f'{SEG_DIR}/metadata.csv',     # small dataset: same split for eval
    output_path=MODEL_DIR,
    max_audio_length=255000,
)

In [ ]:
# Generate test samples
from TTS.api import TTS
import glob, os

checkpoint = sorted(glob.glob(f'{MODEL_DIR}/**/*.pth', recursive=True))[-1]
tts = TTS(model_path=checkpoint, config_path=f'{MODEL_DIR}/config.json').to('cuda')

# Use a real clip as the speaker reference voice
ref_wav = glob.glob(f'{SEG_DIR}/audio/*.wav')[0]

test_sentences = [
    'Nnâmahanga alema amalunga n\'igulu.',
    'Obulangashane bwanaciba.',
    'Nnâmahanga abona oku obulangashane buli bwinjà.',
]

out_dir = f'{DRIVE_ROOT}/results/{TRACK}_samples'
os.makedirs(out_dir, exist_ok=True)

for i, text in enumerate(test_sentences):
    out = f'{out_dir}/sample_{i+1:02d}.wav'
    tts.tts_to_file(text=text, speaker_wav=ref_wav, language='sw', file_path=out)
    print(f'Generated: {out}')